# Inicio

In [43]:
# entrenar_tiny_FIXED.py
"""
Versión corregida con loss combinado
"""

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
import pickle
import gc
import os

from modeloC_tiny import AttentionUNetTiny

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

EPOCHS = 50
BATCH_SIZE = 2
ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
PATIENCE = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memoria: {mem_gb:.2f} GB")
    torch.cuda.empty_cache()
    gc.collect()
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print("\nCargando datos...")

with open('data_minimal.pkl', 'rb') as f:
    data = pickle.load(f)

from dataset_form import train_transform, val_transform, SimpleDataset

train_dataset = SimpleDataset(
    data['train'],
    '../train/',
    data['annotations'],
    train_transform
)

val_dataset = SimpleDataset(
    data['val'],
    '../train/',
    data['annotations'],
    val_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Batch: {BATCH_SIZE} (efectivo: {BATCH_SIZE * ACCUMULATION_STEPS})")


# ========================================
# LOSS CORREGIDO - CON DICE
# ========================================

class CombinedLoss(nn.Module):
    """Loss combinado: Cross Entropy + Dice Loss"""
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=255)
    
    def forward(self, pred, target):
        # Cross Entropy
        ce_loss = self.ce(pred, target)
        
        # Dice Loss
        pred_soft = torch.softmax(pred, dim=1)[:, 1, :, :]
        valid_mask = (target != 255).float()
        target_binary = (target == 1).float()
        
        pred_soft = pred_soft * valid_mask
        target_binary = target_binary * valid_mask
        
        intersection = (pred_soft * target_binary).sum()
        union = pred_soft.sum() + target_binary.sum()
        
        dice = (2.0 * intersection + 1.0) / (union + 1.0)
        dice_loss = 1 - dice
        
        # Limpiar memoria
        del pred_soft, valid_mask, target_binary, intersection, union, dice
        
        return 0.5 * ce_loss + 0.5 * dice_loss


def dice_score(pred, target):
    pred = torch.argmax(pred, dim=1).cpu().numpy()
    target = target.cpu().numpy()
    
    valid = (target != 255)
    pred = pred[valid]
    target = target[valid]
    
    if len(pred) == 0:
        return 0.0
    
    intersection = np.sum(pred * target)
    return (2.0 * intersection) / (np.sum(pred) + np.sum(target) + 1e-7)


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


print("\nCreando modelo...")

model = AttentionUNetTiny(in_channels=3, num_classes=2)
use_amp = torch.cuda.is_available()

# Usar sintaxis nueva para evitar warnings
if use_amp:
    scaler = torch.amp.GradScaler('cuda')
else:
    scaler = None

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# USAR LOSS COMBINADO
criterion = CombinedLoss()

total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"Mixed Precision: {use_amp}")
print(f"Loss: Combined (CE + Dice)")

clear_memory()


print("\nINICIANDO ENTRENAMIENTO")
print("="*50)

best_dice = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    
    # TRAIN
    model.train()
    train_loss = 0
    optimizer.zero_grad()
    
    train_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f'Epoch {epoch+1}/{EPOCHS} [Train]',
        leave=False
    )
    
    for batch_idx, (images, masks, _) in train_bar:
        try:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            if use_amp:
                with torch.amp.autocast('cuda'):  # Sintaxis nueva
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                    loss = loss / ACCUMULATION_STEPS
                
                scaler.scale(loss).backward()
                
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            else:
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss = loss / ACCUMULATION_STEPS
                loss.backward()
                
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad()
            
            train_loss += loss.item() * ACCUMULATION_STEPS
            train_bar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})
            
            del images, masks, outputs, loss
            
            if (batch_idx + 1) % (ACCUMULATION_STEPS * 3) == 0:
                clear_memory()
        
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\nERROR de memoria en batch {batch_idx}")
                clear_memory()
                continue
            else:
                raise e
    
    if use_amp:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    optimizer.zero_grad()
    
    train_loss /= len(train_loader)
    clear_memory()
    
    # VALIDATION
    model.eval()
    val_loss = 0
    val_dice = 0
    val_count = 0
    
    with torch.no_grad():
        val_bar = tqdm(
            val_loader,
            desc=f'Epoch {epoch+1}/{EPOCHS} [Val]',
            leave=False
        )
        
        for images, masks, _ in val_bar:
            try:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                
                if use_amp:
                    with torch.amp.autocast('cuda'):  # Sintaxis nueva
                        outputs = model(images)
                        loss = criterion(outputs, masks)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                
                val_loss += loss.item()
                val_dice += dice_score(outputs, masks)
                val_count += 1
                
                val_bar.set_postfix({'dice': f'{dice_score(outputs, masks):.4f}'})
                
                del images, masks, outputs, loss
            
            except RuntimeError as e:
                if "out of memory" in str(e):
                    clear_memory()
                    continue
                else:
                    raise e
    
    val_loss /= max(val_count, 1)
    val_dice /= max(val_count, 1)
    clear_memory()
    
    # RESULTADOS
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val Dice:   {val_dice:.4f}")
    
    if torch.cuda.is_available():
        mem_alloc = torch.cuda.memory_allocated(0) / 1e9
        mem_res = torch.cuda.memory_reserved(0) / 1e9
        print(f"  GPU:        {mem_alloc:.2f}GB / {mem_res:.2f}GB")
    
    # GUARDAR
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'dice': best_dice
        }, 'mejor_modelo_tiny.pth')
        print(f"  GUARDADO (Dice: {best_dice:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping")
        break
    
    print()

print("\n" + "="*50)
print("COMPLETADO")
print("="*50)
print(f"Mejor Dice: {best_dice:.4f}")
print(f"Archivo: mejor_modelo_tiny.pth")

clear_memory()

Dispositivo: cuda
GPU: NVIDIA GeForce MX330
Memoria: 2.15 GB

Cargando datos...
Train: 4923
Val: 1055
Batch: 2 (efectivo: 16)

Creando modelo...
Parámetros: 487,597 (0.49M)
Mixed Precision: True
Loss: Combined (CE + Dice)

INICIANDO ENTRENAMIENTO



Epoch 1/50
  Train Loss: 0.5842
  Val Loss:   0.5151
  Val Dice:   0.0000
  GPU:        0.01GB / 0.01GB




Epoch 2/50
  Train Loss: 0.3726
  Val Loss:   0.2326
  Val Dice:   0.0000
  GPU:        0.01GB / 0.01GB




Epoch 3/50
  Train Loss: 0.2529
  Val Loss:   0.2200
  Val Dice:   0.0000
  GPU:        0.01GB / 0.01GB




Epoch 4/50
  Train Loss: 0.2448
  Val Loss:   0.2206
  Val Dice:   0.0000
  GPU:        0.01GB / 0.01GB



...


Epoch 12/50
  Train Loss: 0.2123
  Val Loss:   0.1832
  Val Dice:   0.0852
  GPU:        0.01GB / 0.01GB
  GUARDADO (Dice: 0.0852)




Epoch 13/50
  Train Loss: 0.2094
  Val Loss:   0.1790
  Val Dice:   0.0835
  GPU:        0.01GB / 0.01GB




Epoch 14/50
  Train Loss: 0.2033
  Val Loss:   0.1738
  Val Dice:   0.0907
  GPU:        0.01GB / 0.01GB
  GUARDADO (Dice: 0.0907)




Epoch 15/50
  Train Loss: 0.1944
  Val Loss:   0.1849
  Val Dice:   0.1113
  GPU:        0.01GB / 0.01GB
  GUARDADO (Dice: 0.1113)



KeyboardInterrupt: 

In [23]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from modeloC_tiny import AttentionUNetTiny


# Crear modelo y cargar pesos
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AttentionUNetTiny(in_channels=3, num_classes=2).to(device)
model.load_state_dict(torch.load('./mejor_modelo_tiny.pth', weights_only=False)['model_state_dict'])
model.eval()

# Obtener algunas imágenes
images, masks, _ = next(iter(val_loader))
images = images.to(device)

# Hacer predicciones
with torch.no_grad():
    outputs = model(images)
    predictions = torch.argmax(outputs, dim=1)

# Guardar visualizaciones
for i in range(min(4, len(images))):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Imagen original
    img = images[i].cpu().numpy().transpose(1, 2, 0)
    axes[0].imshow(img)
    axes[0].set_title('Original')
    
    # Ground truth
    axes[1].imshow(masks[i].cpu().numpy(), cmap='gray')
    axes[1].set_title('Verdad')
    
    # Predicción
    axes[2].imshow(predictions[i].cpu().numpy(), cmap='gray')
    axes[2].set_title('Predicción')
    
    plt.savefig(f'resultado_{i}.png')
    plt.close()

print("Imágenes guardadas: resultado_0.png, resultado_1.png, ...")

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.0357141..2.6399999].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.0357141..2.6399999].


Imágenes guardadas: resultado_0.png, resultado_1.png, ...


In [22]:
import pickle
import numpy as np

# Cargar datos
with open('data_minimal.pkl', 'rb') as f:
    data = pickle.load(f)

from dataset_form import SimpleDataset, val_transform

# Crear dataset de validación
val_dataset = SimpleDataset(
    data['val'],
    '../train/',
    data['annotations'],
    val_transform
)

print("DIAGNÓSTICO DEL DATASET:")
print("="*60)
print(f"Total imágenes en val: {len(val_dataset)}")
print(f"Total anotaciones en dict: {len(data['annotations'])}")

# Verificar primeras 10 imágenes
print("\nVerificando primeras 10 imágenes:")
for i in range(min(10, len(val_dataset))):
    try:
        img, mask, tile_id = val_dataset[i]
        unique_vals = torch.unique(mask)
        
        print(f"{i}. {tile_id}")
        print(f"   Imagen shape: {img.shape}")
        print(f"   Mask shape: {mask.shape}")
        print(f"   Valores únicos en mask: {unique_vals.numpy()}")
        print(f"   Min/Max: {mask.min().item():.0f} / {mask.max().item():.0f}")
        
        if mask.max() == 0:
            print(f"   ⚠️  MÁSCARA VACÍA!")
        else:
            print(f"   ✓ Tiene anotaciones")
        print()
        
    except Exception as e:
        print(f"{i}. ERROR: {e}\n")

# Verificar estructura de annotations_dict
print("\nEstructura del diccionario de anotaciones:")
sample_keys = list(data['annotations'].keys())[:3]
for key in sample_keys:
    print(f"\nKey: {key}")
    ann = data['annotations'][key]
    print(f"  Tipo: {type(ann)}")
    if isinstance(ann, dict):
        print(f"  Keys: {ann.keys()}")
    elif isinstance(ann, list):
        print(f"  Elementos: {len(ann)}")

DIAGNÓSTICO DEL DATASET:
Total imágenes en val: 1055
Total anotaciones en dict: 1633

Verificando primeras 10 imágenes:
0. fef7ed9d6051
   Imagen shape: torch.Size([3, 512, 512])
   Mask shape: torch.Size([512, 512])
   Valores únicos en mask: [0]
   Min/Max: 0 / 0
   ⚠️  MÁSCARA VACÍA!

1. 0c6ee5bf522b
   Imagen shape: torch.Size([3, 512, 512])
   Mask shape: torch.Size([512, 512])
   Valores únicos en mask: [0]
   Min/Max: 0 / 0
   ⚠️  MÁSCARA VACÍA!

2. 7514d946ca86
   Imagen shape: torch.Size([3, 512, 512])
   Mask shape: torch.Size([512, 512])
   Valores únicos en mask: [  0   1 255]
   Min/Max: 0 / 255
   ✓ Tiene anotaciones

3. cf921187e2bc
   Imagen shape: torch.Size([3, 512, 512])
   Mask shape: torch.Size([512, 512])
   Valores únicos en mask: [0 1]
   Min/Max: 0 / 1
   ✓ Tiene anotaciones

4. d2ab4aae54c4
   Imagen shape: torch.Size([3, 512, 512])
   Mask shape: torch.Size([512, 512])
   Valores únicos en mask: [0]
   Min/Max: 0 / 0
   ⚠️  MÁSCARA VACÍA!

5. ebe2cbbc92e4
   

In [20]:
# Verificar el código actual
with open('dataset_form.py', 'r') as f:
    lines = f.readlines()
    
# Buscar la línea del return
for i, line in enumerate(lines[-10:], start=len(lines)-10):
    print(f"Línea {i}: {line.rstrip()}")

Línea 46:                     coords = np.array(ann['coordinates'], dtype=np.int32)
Línea 47:                     cv2.fillPoly(mask, [coords], 255)  # Ignorar
Línea 48: 
Línea 49:         # Augmentation
Línea 50:         if self.transform:
Línea 51:             augmented = self.transform(image=img, mask=mask)
Línea 52:             img = augmented['image']
Línea 53:             mask = augmented['mask']
Línea 54: 
Línea 55:         return img, mask.long(), tile_id


In [21]:
import sys
import importlib

# Eliminar el módulo del cache
if 'dataset_form' in sys.modules:
    del sys.modules['dataset_form']

# Reimportar
from dataset_form import SimpleDataset, train_transform, val_transform

print("✓ Módulo recargado")

# Ahora ejecuta el diagnóstico de nuevo
import pickle
import torch

with open('data_minimal.pkl', 'rb') as f:
    data = pickle.load(f)

val_dataset = SimpleDataset(
    data['val'],
    '../train/',
    data['annotations'],
    val_transform
)

print("\n🔍 VERIFICACIÓN POST-RECARGA:")
print("="*60)

for i in range(5):
    img, mask, tile_id = val_dataset[i]
    print(f"\n{i}. tile_id = {tile_id}")  # Ahora debería ser un string único
    print(f"   Tipo: {type(tile_id)}")
    print(f"   Mask shape: {mask.shape}")
    print(f"   Valores únicos: {torch.unique(mask).numpy()}")
    print(f"   {'✓ OK' if isinstance(tile_id, str) else '❌ TODAVÍA ES LISTA'}")

✓ Módulo recargado

🔍 VERIFICACIÓN POST-RECARGA:

0. tile_id = fef7ed9d6051
   Tipo: <class 'str'>
   Mask shape: torch.Size([512, 512])
   Valores únicos: [0]
   ✓ OK

1. tile_id = 0c6ee5bf522b
   Tipo: <class 'str'>
   Mask shape: torch.Size([512, 512])
   Valores únicos: [0]
   ✓ OK

2. tile_id = 7514d946ca86
   Tipo: <class 'str'>
   Mask shape: torch.Size([512, 512])
   Valores únicos: [  0   1 255]
   ✓ OK

3. tile_id = cf921187e2bc
   Tipo: <class 'str'>
   Mask shape: torch.Size([512, 512])
   Valores únicos: [0 1]
   ✓ OK

4. tile_id = d2ab4aae54c4
   Tipo: <class 'str'>
   Mask shape: torch.Size([512, 512])
   Valores únicos: [0]
   ✓ OK
